# Analysis Notebook

This notebook supports the narrative "Distance Doesn't Explain Price." It:
1. Loads the cleaned BTS DB1B dataset
2. Computes route-level HHI (carrier concentration)
3. Runs regression comparisons to show market power explains more than distance
4. Exports aggregated JSON files consumed by the report visualizations

## 1. Imports & Data Load

In [3]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

DATA_PATH = '/Volumes/Extreme Pro/DSAN-5200/final-project/T_DB1B_MARKET_CLEAN.csv'
OUT_DIR   = "../../assets/data"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head(3)

Loaded 30,954,956 rows × 19 columns


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE,ORIGIN_LAT,ORIGIN_LON,DEST_LAT,DEST_LON
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2,40.6521,-75.440804,32.411301,-99.681900
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001


In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

DATA_PATH = "../../data/cleandata/T_DB1B_MARKET_CLEAN.csv"
OUT_DIR   = "../../assets/data"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head(3)

Loaded 25,072,315 rows × 19 columns


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE,ORIGIN_LAT,ORIGIN_LON,DEST_LAT,DEST_LON
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2,40.6521,-75.440804,32.411301,-99.681900
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,324.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001


## 2. Compute Route-Level HHI

HHI (Herfindahl-Hirschman Index) measures carrier concentration per route.  
HHI = Σ(market_share²) × 10,000. Range: 0 (perfect competition) → 10,000 (monopoly).

In [ ]:
# Normalize route so ATL->JFK and JFK->ATL are the same market
df["ROUTE"] = df.apply(lambda r: "-".join(sorted([r["ORIGIN"], r["DEST"]])), axis=1)

# Passengers per carrier per route
carrier_pax = (
    df.groupby(["ROUTE", "REPORTING_CARRIER"])["PASSENGERS"]
    .sum()
    .reset_index()
)

# Total passengers per route
route_pax = carrier_pax.groupby("ROUTE")["PASSENGERS"].sum().rename("ROUTE_PAX")
carrier_pax = carrier_pax.join(route_pax, on="ROUTE")
carrier_pax["SHARE"] = carrier_pax["PASSENGERS"] / carrier_pax["ROUTE_PAX"]

# HHI per route
hhi = (
    carrier_pax.groupby("ROUTE")
    .apply(lambda g: (g["SHARE"] ** 2).sum() * 10_000, include_groups=False)
    .rename("HHI")
    .reset_index()
)

print(f"Routes with HHI computed: {len(hhi):,}")
print(hhi["HHI"].describe())
hhi.head()

## 3. Build Route-Level Summary Table

Aggregate to one row per route: mean fare, distance, HHI, top carrier, passenger volume.

In [3]:
# Top carrier per route (by passengers)
top_carrier = (
    carrier_pax.sort_values("PASSENGERS", ascending=False)
    .drop_duplicates("ROUTE")[["ROUTE", "REPORTING_CARRIER", "SHARE"]]
    .rename(columns={"REPORTING_CARRIER": "TOP_CARRIER", "SHARE": "TOP_CARRIER_SHARE"})
)

# Route aggregates
route_agg = (
    df.groupby("ROUTE")
    .agg(
        AVG_FARE=("MARKET_FARE", "mean"),
        MEDIAN_FARE=("MARKET_FARE", "median"),
        AVG_DISTANCE=("MARKET_DISTANCE", "mean"),
        TOTAL_PASSENGERS=("PASSENGERS", "sum"),
        ORIGIN=("ORIGIN", "first"),
        DEST=("DEST", "first"),
    )
    .reset_index()
)

routes = (
    route_agg
    .merge(hhi, on="ROUTE")
    .merge(top_carrier, on="ROUTE")
)

# Only keep routes with enough volume for stable estimates
routes = routes[routes["TOTAL_PASSENGERS"] >= 1000].copy()
routes["LOG_FARE"] = np.log(routes["AVG_FARE"])
routes["LOG_DISTANCE"] = np.log(routes["AVG_DISTANCE"])

print(f"Routes retained (≥1000 pax): {len(routes):,}")
routes.describe()

Routes retained (≥1000 pax): 4,638


,AVG_FARE,MEDIAN_FARE,AVG_DISTANCE,TOTAL_PASSENGERS,HHI,TOP_CARRIER_SHARE,LOG_FARE,LOG_DISTANCE
count,4638.000000,4638.000000,4638.000000,4638.000000,4638.000000,4638.000000,4638.000000,4638.000000
mean,275.613259,254.905277,1192.088751,8538.110608,4762.454962,0.595189,5.587109,6.900149
std,66.319592,67.403959,705.568730,14670.635198,2514.987022,0.228658,0.263164,0.638241
min,57.182702,50.500000,68.112306,1000.000000,1072.741398,0.171911,4.046251,4.221158
25%,233.927599,213.000000,676.111823,1610.250000,2741.080093,0.406454,5.455012,6.516358
50%,273.521037,251.127500,1039.112003,3031.500000,3992.430907,0.551057,5.611379,6.946122
75%,314.629913,294.000000,1593.229252,8240.750000,6333.345248,0.780509,5.751397,7.373518
max,577.562546,584.000000,5123.588028,184571.000000,10000.000000,1.000000,6.358817,8.541610


## 4. Regression Comparison

Compare R² across three models on route-level data:
- **M1**: log(fare) ~ log(distance)
- **M2**: log(fare) ~ log(distance) + HHI
- **M3**: log(fare) ~ HHI only

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

y = routes["LOG_FARE"].values.reshape(-1, 1)

models = {
    "Distance only":          routes[["LOG_DISTANCE"]].values,
    "HHI only":               routes[["HHI"]].values,
    "Distance + HHI":         routes[["LOG_DISTANCE", "HHI"]].values,
}

results = {}
for name, X in models.items():
    lr = LinearRegression().fit(X, y)
    r2 = r2_score(y, lr.predict(X))
    results[name] = round(r2, 4)
    print(f"{name:25s}  R² = {r2:.4f}")

# Save for infographic
with open(f"{OUT_DIR}/r2_comparison.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved r2_comparison.json")

Distance only              R² = 0.4542
HHI only                   R² = 0.1508
Distance + HHI             R² = 0.4946

Saved r2_comparison.json


## 5. Export JSON for Visualizations

In [5]:
# --- 5a. Scatter data: distance vs fare (sample 3000 routes for browser performance)
scatter_cols = ["ROUTE", "AVG_DISTANCE", "AVG_FARE", "HHI", "TOP_CARRIER", "TOTAL_PASSENGERS"]
scatter = routes[scatter_cols].sample(n=min(3000, len(routes)), random_state=42)
scatter.to_json(f"{OUT_DIR}/scatter_distance_fare.json", orient="records", indent=2)
print(f"scatter_distance_fare.json  — {len(scatter)} rows")

# --- 5b. HHI bubble chart: all routes with volume weight
bubble_cols = ["ROUTE", "ORIGIN", "DEST", "AVG_FARE", "HHI", "AVG_DISTANCE",
               "TOP_CARRIER", "TOP_CARRIER_SHARE", "TOTAL_PASSENGERS"]
routes[bubble_cols].to_json(f"{OUT_DIR}/hhi_bubble.json", orient="records", indent=2)
print(f"hhi_bubble.json             — {len(routes)} rows")

# --- 5c. Top 20 monopoly routes (HHI > 7500, high fare)
monopoly = (
    routes[routes["HHI"] >= 7500]
    .nlargest(20, "AVG_FARE")[bubble_cols]
)
monopoly.to_json(f"{OUT_DIR}/monopoly_routes.json", orient="records", indent=2)
print(f"monopoly_routes.json        — {len(monopoly)} rows")

# --- 5d. Airport carrier concentration (for linked map → bar chart)
# Per airport: which carriers fly out, what share of passengers
origin_carrier = (
    df.groupby(["ORIGIN", "REPORTING_CARRIER"])["PASSENGERS"]
    .sum()
    .reset_index()
)
origin_total = origin_carrier.groupby("ORIGIN")["PASSENGERS"].sum().rename("ORIGIN_PAX")
origin_carrier = origin_carrier.join(origin_total, on="ORIGIN")
origin_carrier["SHARE"] = origin_carrier["PASSENGERS"] / origin_carrier["ORIGIN_PAX"]

# Airport lat/lon
airport_coords = (
    df.groupby("ORIGIN")
    .agg(LAT=("ORIGIN_LAT","first"), LON=("ORIGIN_LON","first"), TOTAL_PAX=("PASSENGERS","sum"))
    .reset_index()
    .rename(columns={"ORIGIN":"AIRPORT"})
)

# Build nested dict: {airport: [{carrier, share}, ...]}
airport_carriers = (
    origin_carrier
    .sort_values(["ORIGIN","PASSENGERS"], ascending=[True, False])
    .groupby("ORIGIN")
    .apply(lambda g: g[["REPORTING_CARRIER","SHARE"]].rename(
        columns={"REPORTING_CARRIER":"carrier","SHARE":"share"}).to_dict("records"),
        include_groups=False)
    .reset_index()
    .rename(columns={"ORIGIN":"AIRPORT", 0:"carriers"})
)

airport_map = airport_coords.merge(airport_carriers, on="AIRPORT")
airport_map["carriers"] = airport_map["carriers"].apply(lambda x: x[:8])  # top 8 carriers only
airport_map.to_json(f"{OUT_DIR}/airport_carriers.json", orient="records", indent=2)
print(f"airport_carriers.json       — {len(airport_map)} airports")

scatter_distance_fare.json  — 3000 rows
hhi_bubble.json             — 4638 rows
monopoly_routes.json        — 20 rows
airport_carriers.json       — 447 airports
